# Agentic AI Demo - Project Scaffolding Agent with Gemini + MCP

An **AI-powered project scaffolding agent** built with **Google Gemini 2.5 Flash Lite** and the **Model Context Protocol (MCP)**.
Give it a project style and a topic, and it autonomously generates a complete folder structure with real, functional starter code.

### What makes this Agentic AI?
This agent operates in a **true agentic loop** - not a single-shot prompt-and-execute:

- **Observe → Think → Act → Observe loop** - The agent runs iteratively, deciding one action at a time based on feedback
- **MCP Tool Server** - Tools (`create_folder`, `create_file`, `write_to_file`) are exposed via a **Model Context Protocol server**, discovered at runtime - not hardcoded
- **Gemini Function Calling** - Gemini natively calls tools using structured function calls - no JSON parsing or `eval()` needed
- **Autonomy** - Given only a style and topic, the agent independently decides everything
- **Self-correction** - Errors are fed back to the agent, which can adapt and retry
- **Memory** - Full conversation history across steps

> **TL;DR:** A regular AI *tells* you what to do. An agentic AI *does* it - step by step, with MCP tools and feedback.

### Architecture
```
┌─────────────┐       function call     ┌──────────────┐      JSON-RPC/stdio    ┌──────────────┐
│  Gemini LLM │ <────────────────────── │  Agent Loop  │ ──────────────────────>│  MCP Server  │
│  (thinks)   │ ──────────────────────> │(orchestrates)│ <──────────────────────│  (tools)     │
└─────────────┘          result         └──────────────┘         result         └──────────────┘
```

### Supported Styles
`rest-api` · `cli-tool` · `web-app` · `fullstack` · `data-science` · `ml-pipeline` · `python-package` · `chrome-extension` · `discord-bot` · `mobile-app` · `microservice` · `static-site` · `game` · `vscode-extension` · `terraform`

In [25]:
!pip install -q -U google-genai python-dotenv mcp nest-asyncio

import os, sys, ast, json, time
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai.errors import ClientError
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import nest_asyncio

nest_asyncio.apply()
load_dotenv()

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MCP_SERVER = os.path.join(os.environ["OUTPUT_DIR"].rsplit("/output", 1)[0], "os-mcp.py")
print(f"MCP server: {MCP_SERVER}")

MCP server: /Users/arantarokade/Documents/speedrun-gen-ai/os-mcp.py


In [26]:
# Supported project styles
PROJECT_STYLES = {
    "rest-api":          "REST API backend (Flask/FastAPI) with routes, models, middleware",
    "cli-tool":          "Command-line tool with argument parsing and subcommands",
    "web-app":           "Frontend web app (HTML/CSS/JS or React/Vue)",
    "fullstack":         "Full-stack app with frontend + backend + database",
    "data-science":      "Data science project with notebooks, data, and scripts",
    "ml-pipeline":       "Machine learning pipeline with training, evaluation, and inference",
    "python-package":    "Publishable Python package with setup.py, tests, and docs",
    "chrome-extension":  "Chrome browser extension with manifest, popup, and background scripts",
    "discord-bot":       "Discord bot with commands, events, and cogs",
    "mobile-app":        "Mobile app (React Native / Flutter) with screens and navigation",
    "microservice":      "Microservice with Docker, health checks, and message queue support",
    "static-site":       "Static site / blog with templates and content",
    "game":              "Simple game project (Pygame / JS Canvas) with assets and scenes",
    "vscode-extension":  "VS Code extension with commands, views, and settings",
    "terraform":         "Infrastructure-as-Code project with Terraform modules and envs",
}

print("Supported project styles:\n")
for key, desc in PROJECT_STYLES.items():
    print(f"  • {key:20s} → {desc}")

Supported project styles:

  • rest-api             → REST API backend (Flask/FastAPI) with routes, models, middleware
  • cli-tool             → Command-line tool with argument parsing and subcommands
  • web-app              → Frontend web app (HTML/CSS/JS or React/Vue)
  • fullstack            → Full-stack app with frontend + backend + database
  • data-science         → Data science project with notebooks, data, and scripts
  • ml-pipeline          → Machine learning pipeline with training, evaluation, and inference
  • python-package       → Publishable Python package with setup.py, tests, and docs
  • chrome-extension     → Chrome browser extension with manifest, popup, and background scripts
  • discord-bot          → Discord bot with commands, events, and cogs
  • mobile-app           → Mobile app (React Native / Flutter) with screens and navigation
  • microservice         → Microservice with Docker, health checks, and message queue support
  • static-site          → Static si

In [27]:
SYSTEM_PROMPT = """You are an expert software architect agent. You scaffold projects ONE STEP AT A TIME using the provided tools.

Rules:
- ALWAYS call exactly ONE tool per turn — never reply with just text unless you are completely finished
- Start by creating the root project folder, then build out subfolders and files
- Write REAL, functional starter code — not placeholders or TODOs
- Include config files (.gitignore, Dockerfile, etc.) relevant to the project style
- Include a README.md with setup instructions
- Include requirements.txt or package.json with real dependencies
- Use best practices for the given project style
- ONLY when ALL files are created, respond with a brief text summary (no tool call) listing every file you created
- Do NOT explain what you will do — just do it by calling a tool
"""

In [28]:
# ── Verify MCP server works ────────────────────────────────────────────
server_params = StdioServerParameters(
    command=sys.executable,
    args=[MCP_SERVER],
)

# Quick connect → list tools → disconnect to verify the server is working.
async def verify_mcp():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = (await session.list_tools()).tools
            print(f"MCP tools discovered: {[t.name for t in mcp_tools]}")
            for t in mcp_tools:
                print(f"\t -{t.name}: {t.description}")
                print(f"\t\t Parameters: {t.inputSchema.items()}")

    print("MCP server verified and ready")

await verify_mcp()

MCP tools discovered: ['create_folder', 'create_file', 'write_to_file']
	 -create_folder: Create a directory and all parent directories at the given path.
		 Parameters: dict_items([('properties', {'folder_path': {'title': 'Folder Path', 'type': 'string'}}), ('required', ['folder_path']), ('title', 'create_folderArguments'), ('type', 'object')])
	 -create_file: Create an empty file at the given path (parent directories are created automatically).
		 Parameters: dict_items([('properties', {'file_path': {'title': 'File Path', 'type': 'string'}}), ('required', ['file_path']), ('title', 'create_fileArguments'), ('type', 'object')])
	 -write_to_file: Write text content to a file, creating or overwriting it (parent directories are created automatically).
		 Parameters: dict_items([('properties', {'file_path': {'title': 'File Path', 'type': 'string'}, 'file_content': {'title': 'File Content', 'type': 'string'}}), ('required', ['file_path', 'file_content']), ('title', 'write_to_fileArguments')

In [29]:
MAX_STEPS = 50
REQUEST_DELAY = 7   # seconds between Gemini calls (free tier: 10 req/min)
MAX_RETRIES = 3     # retries on rate-limit (429) errors

async def scaffold_project(style: str, topic: str):
    """Agentic loop: connects to MCP, discovers tools, uses Gemini function calling."""
    if style not in PROJECT_STYLES:
        print(f"Unknown style '{style}'. Choose from: {', '.join(PROJECT_STYLES.keys())}")
        return

    OUTPUT_DIR = os.environ["OUTPUT_DIR"]
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.chdir(OUTPUT_DIR)

    # ── Connect to MCP server ─────────────────────────────────────────
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Discover tools & convert to Gemini function declarations
            mcp_tools = (await session.list_tools()).tools
            gemini_tools = types.Tool(function_declarations=[
                types.FunctionDeclaration(
                    name=tool.name,
                    description=tool.description,
                    parameters={
                        k: v for k, v in tool.inputSchema.items()
                        if k not in ("additionalProperties", "$schema")
                    },
                )
                for tool in mcp_tools
            ])

            # Start conversation
            contents = [
                types.Content(role="user", parts=[
                    types.Part.from_text(text=
                        f'Create a **{style}** project about: "{topic}"\n'
                        f'Style description: {PROJECT_STYLES[style]}\n'
                        f'Scaffold the project step by step.'
                    )
                ])
            ]

            print(f"Agent started: {style} project -- \"{topic}\"")
            print(f"Output dir: {OUTPUT_DIR}\n")

            # ── Agentic loop ──────────────────────────────────────────
            for step in range(1, MAX_STEPS + 1):

                # THINK: ask Gemini (with rate-limit retry)
                response = None
                for attempt in range(1, MAX_RETRIES + 1):
                    try:
                        time.sleep(REQUEST_DELAY)
                        response = client.models.generate_content(
                            model="gemini-2.5-flash-lite",
                            contents=contents,
                            config=types.GenerateContentConfig(
                                system_instruction=SYSTEM_PROMPT,
                                tools=[gemini_tools],
                                temperature=0.7,
                            ),
                        )
                        break  # success
                    except ClientError as e:
                        if "429" in str(e) and attempt < MAX_RETRIES:
                            wait = 35 * attempt
                            print(f"  Step {step}: [RATE LIMIT] Waiting {wait}s (attempt {attempt}/{MAX_RETRIES})...")
                            time.sleep(wait)
                        else:
                            raise

                # Guard against empty / blocked responses
                candidate = response.candidates[0] if response.candidates else None
                if not candidate or not candidate.content or not candidate.content.parts:
                    reason = getattr(candidate, "finish_reason", "unknown") if candidate else "no candidates"
                    print(f"  Step {step}: [SKIP] Empty response (reason: {reason}), nudging model...")
                    contents.append(types.Content(role="user", parts=[
                        types.Part.from_text(text="Continue with the next step.")
                    ]))
                    continue

                contents.append(candidate.content)
                part = candidate.content.parts[0]

                # DONE? (text reply = no more tool calls)
                fc = getattr(part, "function_call", None)
                if not fc or not fc.name:
                    # Too early — model is "planning" instead of acting, nudge it
                    if step < 5:
                        print(f"  Step {step}: [NUDGE] Model replied with text too early, pushing to act...")
                        contents.append(types.Content(role="user", parts=[
                            types.Part.from_text(text="Don't explain — call a tool now. Start creating the project.")
                        ]))
                        continue
                    print(f"\n[DONE] Agent finished in {step} steps!")
                    print(f"  {part.text}")
                    print(f"  Project at: {OUTPUT_DIR}")
                    return

                # ACT: execute tool via MCP
                fn_name = fc.name
                fn_args = dict(fc.args) if fc.args else {}

                try:
                    if fn_name == "write_to_file":
                        print(f"  Step {step}: write_to_file('{fn_args.get('file_path', '')}')")
                    else:
                        arg_val = next(iter(fn_args.values()), "")
                        print(f"  Step {step}: {fn_name}('{arg_val}')")

                    result = await session.call_tool(fn_name, fn_args)
                    result_text = result.content[0].text if result.content else "Done"
                    print(f"           > {result_text}")
                except Exception as e:
                    result_text = f"ERROR: {e}"
                    print(f"           [ERR] {e}")

                # OBSERVE: feed result back to Gemini
                contents.append(types.Content(
                    parts=[types.Part.from_function_response(
                        name=fn_name,
                        response={"result": result_text},
                    )]
                ))

            print(f"\n[WARN] Agent hit max steps ({MAX_STEPS}).")

In [ ]:
# Run the agent
await scaffold_project("rest-api", "a bookstore inventory management system")

Agent started: rest-api project -- "a bookstore inventory management system"
Output dir: /Users/arantarokade/Documents/speedrun-gen-ai/output

  Step 1: create_folder('bookstore-inventory-management-system')
           > Folder created: bookstore-inventory-management-system
  Step 2: [SKIP] Empty response (reason: STOP), nudging model...
  Step 3: [RATE LIMIT] Waiting 35s (attempt 1/3)...
  Step 3: write_to_file('bookstore-inventory-management-system/requirements.txt')
           > Content written to: bookstore-inventory-management-system/requirements.txt
  Step 4: [RATE LIMIT] Waiting 35s (attempt 1/3)...
  Step 4: [SKIP] Empty response (reason: STOP), nudging model...


In [ ]:
# ── More examples (uncomment any to try) ──────────────────────────────

# await scaffold_project("cli-tool",          "a file duplicate finder")
# await scaffold_project("fullstack",         "a todo app with authentication")
# await scaffold_project("ml-pipeline",       "sentiment analysis on product reviews")
# await scaffold_project("discord-bot",       "a music trivia quiz bot")
# await scaffold_project("chrome-extension",  "a website reading time estimator")
# await scaffold_project("microservice",      "a URL shortener service")
# await scaffold_project("data-science",      "analyzing global CO2 emissions")
# await scaffold_project("python-package",    "a markdown-to-HTML converter library")
# await scaffold_project("terraform",         "AWS ECS deployment with ALB")
# await scaffold_project("game",              "a snake game with power-ups")
# await scaffold_project("vscode-extension",  "a code snippet manager")
# await scaffold_project("mobile-app",        "a habit tracker with streaks")
# await scaffold_project("static-site",       "a personal developer portfolio")